In [3]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, f1_score
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" # To download it fast trick

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

CSV_PATH = "../data/fd_dataset_messy.csv"
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["subject", "content", "label"]).reset_index(drop=True)
print(df.head(5))


                         email                 time        subject  \
0  mukesh.bhatt@rediffmail.com  2025-02-17 20:29:47  Payment issue   
1      sanjay.jain@hotmail.com  2024-06-07 16:22:21           Help   
2         ashok.nair@gmail.com  2024-04-05 08:16:39     Loan query   
3    manoj.gupta51@hotmail.com  2025-01-04 23:24:05          Query   
4     vinod.chopra87@gmail.com  2025-03-03 10:12:43    Loan status   

                                             content              label  
0  Branch gaya tha, unhone email karne bola. Two ...             Non-FD  
1  Dear customer care, My insurance policy is rea...             Non-FD  
2  Namaskar, Rs 6 lakh kat gaya bina bataye insur...             Non-FD  
3  Dear customer care, 1. Suna hai Bajaj Finance ...  Multiple Category  
4  Dear customer care, Where is my money? The rev...             Non-FD  


In [4]:
# Multilingual BERT handles the Hindi+English (Hinglish) text in this dataset
# much better than plain English BERT. Swap to "bert-base-uncased" if your
# text is actually pure English.
#MODEL_NAME = "bert-base-multilingual-cased"
MODEL_NAME = "distilbert-base-uncased" 
MAX_LENGTH = 128
OUTPUT_DIR = "./bert_fd_classifier"
SEED = 42

In [5]:
# ---------------------------------------------------------------------------
# 1. Load & clean data
# ---------------------------------------------------------------------------
# Combine subject + content into a single text field for the model
df["text"] = (
    df["subject"].fillna("").str.strip() + ". " + df["content"].fillna("").str.strip()
).str.strip()
df = df[df["text"].str.len() > 0].reset_index(drop=True)
print(df.head(5))

                         email                 time        subject  \
0  mukesh.bhatt@rediffmail.com  2025-02-17 20:29:47  Payment issue   
1      sanjay.jain@hotmail.com  2024-06-07 16:22:21           Help   
2         ashok.nair@gmail.com  2024-04-05 08:16:39     Loan query   
3    manoj.gupta51@hotmail.com  2025-01-04 23:24:05          Query   
4     vinod.chopra87@gmail.com  2025-03-03 10:12:43    Loan status   

                                             content              label  \
0  Branch gaya tha, unhone email karne bola. Two ...             Non-FD   
1  Dear customer care, My insurance policy is rea...             Non-FD   
2  Namaskar, Rs 6 lakh kat gaya bina bataye insur...             Non-FD   
3  Dear customer care, 1. Suna hai Bajaj Finance ...  Multiple Category   
4  Dear customer care, Where is my money? The rev...             Non-FD   

                                                text  
0  Payment issue. Branch gaya tha, unhone email k...  
1  Help. Dear cust

In [6]:
# Encode string labels -> integer ids
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])
num_labels = len(label_encoder.classes_)
print("Classes:", list(label_encoder.classes_))
print(df["label"].value_counts())

Classes: ['FD', 'Multiple Category', 'Non-FD']
label
FD                   1002
Multiple Category     750
Non-FD                748
Name: count, dtype: int64


In [7]:
print(df.head(5))

                         email                 time        subject  \
0  mukesh.bhatt@rediffmail.com  2025-02-17 20:29:47  Payment issue   
1      sanjay.jain@hotmail.com  2024-06-07 16:22:21           Help   
2         ashok.nair@gmail.com  2024-04-05 08:16:39     Loan query   
3    manoj.gupta51@hotmail.com  2025-01-04 23:24:05          Query   
4     vinod.chopra87@gmail.com  2025-03-03 10:12:43    Loan status   

                                             content              label  \
0  Branch gaya tha, unhone email karne bola. Two ...             Non-FD   
1  Dear customer care, My insurance policy is rea...             Non-FD   
2  Namaskar, Rs 6 lakh kat gaya bina bataye insur...             Non-FD   
3  Dear customer care, 1. Suna hai Bajaj Finance ...  Multiple Category   
4  Dear customer care, Where is my money? The rev...             Non-FD   

                                                text  label_id  
0  Payment issue. Branch gaya tha, unhone email k...         2 

In [8]:
# ---------------------------------------------------------------------------
# 2. Train / val / test split (stratified so class ratios stay balanced)
# ---------------------------------------------------------------------------
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_id"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=SEED
)
print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

Train: 1750  Val: 375  Test: 375


In [9]:
# ---------------------------------------------------------------------------
# 3. Tokenizer & Dataset
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class FDDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        item = {k: torch.tensor(v) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = FDDataset(train_df["text"], train_df["label_id"], tokenizer)
val_dataset = FDDataset(val_df["text"], val_df["label_id"], tokenizer)
test_dataset = FDDataset(test_df["text"], test_df["label_id"], tokenizer)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

##### AutoTokenizer
- AutoTokenizer is just a dispatcher — it looks at MODEL_NAME's saved tokenizer config and picks the matching tokenizer class. 
- For any BERT checkpoint (including bert-base-multilingual-cased), that resolves to BertTokenizerFast, which implements WordPiece.
- WordPiece works by greedily breaking words into the largest known subword chunks from its vocabulary. 
- If a word isn't in the vocab whole, it splits it — continuation pieces get a ## prefix (e.g. "pareshan" → ["pares", "##han"] or similar, depending on the vocab).
- This is exactly why multilingual BERT handles your Hinglish text reasonably well even without a dedicated Hindi tokenizer: romanized Hindi words that aren't whole vocabulary entries still get decomposed into known subword fragments rather than turning into a single [UNK] token.

NOTE: Other architectures differ here — GPT-2 uses BPE, T5/ALBERT/XLNet use SentencePiece — so if you ever swap MODEL_NAME to one of those, AutoTokenizer would transparently load that scheme instead. The rest of your code wouldn't need to change.

In [10]:
# ---------------------------------------------------------------------------
# 4. Model
# ---------------------------------------------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)
model

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


##### AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
- Same dispatcher pattern as AutoTokenizer, but for the model: looks at MODEL_NAME and loads BertForSequenceClassification with the pretrained BERT weights (embeddings + 12 transformer encoder layers + pooler).
- Key part: num_labels=3 tells it to attach a new linear classification head on top of BERT's pooled output 
- A Linear(hidden_size, 3) layer that maps BERT's 768-dim sentence representation to 3 logits (one per class: FD / Non-FD / Multiple Category). 
- The pretrained checkpoint has no such head (it was pretrained for masked-language-modeling, not your task), so this layer is created fresh with random weights — it hasn't learned anything yet. 
- That's expected; it's what trainer.train() actually trains.

In [11]:
# ---------------------------------------------------------------------------
# 5. Metrics
# ---------------------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [12]:
# ---------------------------------------------------------------------------
# 6. Training
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=20,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


##### TrainingArguments — settings for how training runs:
- output_dir — where checkpoints get saved.
- num_train_epochs=4 — go through the full training set 4 times.
- per_device_train_batch_size=16 / eval_batch_size=32 — how many examples processed at once (eval batch can be bigger since no gradients are stored, so it uses less memory).
- learning_rate=2e-5 — how big each weight-update step is. This small value is the standard choice for fine-tuning BERT (a bigger one can destroy the pretrained knowledge).
- weight_decay=0.01 — a small penalty that discourages weights from growing too large, helps avoid overfitting.
- eval_strategy="epoch" / save_strategy="epoch" — run evaluation, and save a checkpoint, after every epoch (instead of every N steps).
- load_best_model_at_end=True — after all 4 epochs, don't keep the last checkpoint, keep whichever epoch scored best.
metric_for_best_model="f1_macro" — "best" is judged by macro-F1 (matches your compute_metrics function from earlier).
- logging_steps=20 — print training loss every 20 steps, just so you can watch progress.
- seed=SEED — fixes randomness (shuffling, weight init) so results are reproducible.
- report_to="none" — don't send logs to external trackers like Weights & Biases.

##### Trainer — the actual engine that runs the training loop. You hand it everything it needs:
- model — the BERT model with the classifier head.
- args — the settings above.
- train_dataset / eval_dataset — your data.
- data_collator — does the per-batch dynamic padding (from earlier).
- compute_metrics — calculates accuracy/F1 at each evaluation.

Once built, trainer knows everything it needs — calling trainer.train() (next section) is what actually starts the loop: forward pass → loss → backward pass → update weights, repeated for 4 epochs.

In [13]:
if __name__ == "__main__":
    trainer.train()

    # -----------------------------------------------------------------
    # 7. Evaluate on the held-out test set
    # -----------------------------------------------------------------
    test_results = trainer.predict(test_dataset)
    test_preds = np.argmax(test_results.predictions, axis=-1)
    print("\n=== Test set report ===")
    print(
        classification_report(
            test_df["label_id"], test_preds, target_names=label_encoder.classes_
        )
    )

    # -----------------------------------------------------------------
    # 8. Save model + tokenizer + label encoder
    # -----------------------------------------------------------------
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    import joblib

    joblib.dump(label_encoder, f"{OUTPUT_DIR}/label_encoder.joblib")
    print(f"Model saved to {OUTPUT_DIR}")

    # -----------------------------------------------------------------
    # 9. Quick inference helper
    # -----------------------------------------------------------------
    def predict(text, model, tokenizer, label_encoder, device="cpu"):
        model.eval()
        model.to(device)
        inputs = tokenizer(
            text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
        ).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        pred_id = torch.argmax(logits, dim=-1).item()
        return label_encoder.inverse_transform([pred_id])[0]

    sample = test_df["text"].iloc[0]
    print("\nSample prediction:")
    print("Text:", sample)
    print("Predicted:", predict(sample, model, tokenizer, label_encoder))
    print("Actual:", label_encoder.inverse_transform([test_df["label_id"].iloc[0]])[0])

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.324200,0.215489,0.949333,0.950878
2,0.104000,0.175057,0.952000,0.953600
3,0.113600,0.166025,0.962667,0.963830
4,0.113800,0.167790,0.965333,0.966488



=== Test set report ===
                   precision    recall  f1-score   support

               FD       0.96      0.98      0.97       151
Multiple Category       1.00      1.00      1.00       112
           Non-FD       0.97      0.95      0.96       112

         accuracy                           0.98       375
        macro avg       0.98      0.98      0.98       375
     weighted avg       0.98      0.98      0.98       375

Model saved to ./bert_fd_classifier

Sample prediction:
Text: Regarding my investment with you. Hello, Paisa daala tha 12 lakh kuch time pehle. Ab uspe kitna ban gaya hoga? Aur agar abhi nikalna ho toh kya hoga? Suresh Jain
Predicted: FD
Actual: FD


1. Epoch table — training progress
- Training Loss — how wrong the model was on training batches that epoch; lower is better.
- Validation Loss — same loss, but on val_dataset, data it never trains on — the honest signal.
- Accuracy / F1 Macro — from your compute_metrics() function, also computed on validation data.
- Trend: loss drops sharply epoch 1→2 (0.32 → 0.10) as the model learns the basics, then validation loss keeps improving through epoch 3 (0.166) but ticks up slightly at epoch 4 (0.168) — while accuracy/F1 still improve.
- That small uptick is a normal early sign of starting to specialize on training patterns, but mild — both other metrics are still climbing.
- Since you set metric_for_best_model="f1_macro", the Trainer keeps whichever epoch scored highest on F1 (epoch 4, 0.9665), not whichever had the lowest loss.
- That's the checkpoint load_best_model_at_end=True restored before saving.

2. Test set report — final, unbiased score
- This is classification_report() run on test_dataset — data the model never saw during training or checkpoint selection.
- That two-step separation (val picks the best epoch, test reports the final number) is what makes this number trustworthy.
- precision — of everything labeled FD, how much was actually FD (0.96 → a few false positives).
- recall — of everything truly FD, how much got caught (0.98 → very few missed).
- support — real test examples per class (151 / 112 / 112 — matches your 15%-of-2500 split).
- macro avg — average of the three classes' scores, each weighted equally regardless of class size.
- weighted avg — same average, weighted by support instead. Since classes are fairly balanced here, both land at 0.98 — they'd diverge more if one class were tiny.
- Multiple Category hitting a clean 1.00 suggests those emails have a distinctive pattern (likely ones explicitly mixing FD + non-FD complaints) — though with only 112 test examples, worth treating as "looks strong" rather than "guaranteed always perfect."

3. Sample prediction — your predict() function in action
- The complaint mentions investing ₹12 lakh ("Paisa daala") and asks what it's grown to / what happens on withdrawal ("nikalna") — language the model has clearly learned to associate with FD.
- It predicted correctly.
- Mechanically, same decode flow as before: tokenize → model → logits → argmax → inverse_transform → "FD".

Overall
- 98% test accuracy is genuinely strong for a 2500-row dataset.
- Likely explanation: the three classes have fairly distinct vocabulary — FD account numbers, fixed-deposit-specific terms like "matured"/"nikalna" vs. generic loan/insurance complaint language in Non-FD.